# Agent 第 11 课：主流框架全景（Phase 2 开篇）

Phase 1 我们手写了整套 agent 内核：循环、工具路由、记忆、planning、reflection、multi-agent、评测、安全。

现在从"造轮子"切到"读别人的轮子"——看三个有代表性的开源框架把这些组件做成什么样，然后理解**什么时候该用框架、什么时候该自己写**。

> 本课不跑框架代码（依赖太杂），重点是**把我们手写的那些概念映射到框架里的术语**。真正的上手，会放到后面 AWS Bedrock 章节里。

## 1. 三个代表性框架

| 框架 | 定位 | 心智模型 | 谁在用 |
|---|---|---|---|
| **LangGraph**（LangChain 团队） | **有状态图**：把 agent 当作一张 state graph | node = 函数 / LLM 调用，edge = 条件路由，state 一直在传 | 复杂工作流、需要可视化、需要断点恢复 |
| **smolagents**（HuggingFace） | **Code-as-action**：让 LLM 直接写 Python 代码当 action | LLM 输出 `<code>...</code>`，框架在沙箱里执行 | 研究 / 快速原型 / 工具种类多变 |
| **AutoGen**（Microsoft） | **多 agent 对话**：一群 agent 互相发消息 | 每个 agent 是一个 role，消息在群组里流转 | multi-agent 场景、角色扮演、讨论式推理 |

外加厂商托管的：**AWS Bedrock Agents**、**OpenAI Assistants API**——下一课开始讲前者。

## 2. 把我们写过的东西映射过去

Phase 1 你手写的每个概念，框架里都叫别的名字。看一眼就熟了：

| 手写时的概念（Phase 1） | LangGraph | smolagents | AutoGen |
|---|---|---|---|
| `while step < max_steps` 主循环 | `graph.compile().invoke()` | `agent.run(task)` | `initiate_chat()` |
| Thought / Action / Observation | node 之间的 state transitions | 每一步 LLM 生成的 code + 执行结果 | agent 之间的 message |
| Tool dispatcher | `ToolNode` | `@tool` 装饰器 | `register_for_llm()` |
| 短期记忆（messages list） | state 里的一个字段 | `agent.memory` | conversation history |
| 长期记忆 | 外接 vector store（自己接） | 外接 | 外接 |
| Planner | 前置一个 "planner" node | 一个 system prompt 让它先 plan | PlannerAgent 角色 |
| Reflection | 加一个 "critic" node 做条件路由 | 在 code 里自行 assert + 重试 | CriticAgent 角色 |
| Multi-agent | subgraph | 嵌套 `ManagedAgent` | 原生能力 |

**重点**：框架没发明新东西，只是把你写过的循环/路由/记忆做成了 API。

## 3. LangGraph 的核心：State + Graph

LangGraph 的一切都围绕一个 **TypedDict State**：
```python
class State(TypedDict):
    messages: list
    plan: list[str]
    step: int
```

你定义若干 node（函数：`state -> state`），用 edges 连起来：
```python
graph = StateGraph(State)
graph.add_node("planner",  planner_fn)
graph.add_node("executor", executor_fn)
graph.add_node("critic",   critic_fn)

graph.add_edge("planner", "executor")
graph.add_conditional_edges("executor", route_fn, {
    "retry": "executor", "critique": "critic", "done": END
})
```

**关键优点**：状态显式，一切都能 checkpoint → 可中断、可回放、可多 session。这是 Phase 1 我们手写的循环最缺的东西。

**关键代价**：图越画越大，debug 变成看可视化。工作流简单的时候是过度工程。

## 4. smolagents 的核心：Code-as-Action

传统 tool use 是 LLM 输出 JSON，框架 parse、查表、调函数。
smolagents 翻了一层：LLM 直接输出 Python 代码，框架在 **受限解释器** 里执行。

```
Thought: 我需要查今天的汇率然后换算
Code:
```py
rate = get_fx_rate("USD", "CNY")
result = 100 * rate
print(result)
```
```

**好处**：
- 一次 LLM 调用能做**多步逻辑**（循环、条件、组合工具）
- 工具组合无需框架设计 DSL——Python 就是 DSL

**代价**：
- 沙箱安全是大坑（`eval`/`exec` 的那一套风险全回来了）
- 小模型写不好代码就全崩了
- 成本模型里 code 步骤比 JSON tool call 更长

smolagents 提供 `LocalPythonInterpreter` 和 `E2BExecutor`（云沙箱）两种隔离级别。

## 5. AutoGen 的核心：Agent 之间发消息

AutoGen 把 Phase 1 第 8 课的 Manager/Worker 模式做成了**一等公民**：

```python
user   = UserProxyAgent("user")
coder  = AssistantAgent("coder",  system_message="You write code")
critic = AssistantAgent("critic", system_message="You find bugs")

user.initiate_chat(coder, message="Write a fibonacci function")
# coder 说完，critic 可以接话，轮流直到 TERMINATE
```

**用得好的场景**：
- 辩论 / 多视角评审
- 模拟社交情境（研究用途）
- 一个 Planner + 多个 Executor 的工业流水线

**踩过的坑**：
- Token 爆炸——每个 agent 都保留自己视角的完整对话
- 收敛困难——两个 agent 客气半天不结束
- 调试地狱——出错了要回溯三四条 agent 的内部状态

用 AutoGen 要有**严格的终止条件 + token budget**，否则就是第 10 课里的"失控 loop"生产环境版本。

## 6. 什么时候该用框架，什么时候自己写

这是你作为工程师要回答的真问题。我的经验公式：

**自己写（就像 Phase 1）** —— 当满足以下任一条：
- 工具 ≤ 5 个，流程线性或简单分支
- 对延迟 / token 成本敏感，每一个 LLM 调用都要人肉优化
- 需要深度定制某一层（自定义 memory store、自定义 cost accounting）
- 团队没人熟悉框架，引入学习成本 > 节约的代码

**用框架** —— 当满足以下任一条：
- 工作流复杂：多分支 + 循环 + 人机协作断点（→ LangGraph）
- 工具种类爆炸或需要动态组合（→ smolagents）
- 真正的多 agent 协作场景，不是一个 agent 加点 role prompt 能糊过去的（→ AutoGen）
- 团队里多人要改 agent 逻辑，需要一个共同语言

**用厂商托管（Bedrock Agents / OpenAI Assistants）** —— 当：
- 你不想管基础设施（向量库、session 存储、日志）
- 安全合规有要求（企业内部）
- 下一课开始就是这条路线

## 7. 一个反直觉的观察

看完三个框架你会发现：**它们帮你省的是工程样板，不是 agent 智能**。

Agent 的"聪明"几乎 100% 来自：
1. **模型本身的能力**（Claude 4.x / GPT 级别以上才有真正可用的 tool use）
2. **Prompt 的结构化程度**（system prompt + tool description + examples）
3. **工具设计的质量**（工具粒度、输入输出格式）

框架不能让一个笨模型变聪明。**先把 Phase 1 内核啃透**，再看框架就是"哦，这里只是把我的循环改名叫 Graph 了"。

---

## 8. 工程直觉

1. **先写一版手搓的**，再决定上不上框架。直接上框架你会把框架 bug 当成 agent bug。
2. **框架锁死是真的**。LangGraph 的 state 结构一旦传给上游就不好改。选之前看团队能不能一直维护。
3. **Code-as-action（smolagents）只在强模型上好用**。小模型写 Python 经常写错变量名。
4. **Multi-agent 不是越多越好**。90% 的"multi-agent"用一个 agent + 几条 role 切换就够了。
5. **下一课开始进 AWS**：我们用 Bedrock 的 Converse API 重走一遍 tool use，你会看到 AWS 的抽象和 OpenAI 几乎一一对应。

## 下一课预告：Bedrock Converse API + Tool Use

- 为什么 AWS 要出一个自己的 `converse()` 而不是直接兼容 OpenAI
- Converse 的 message 格式 vs OpenAI chat format 的 diff
- 用 Claude Sonnet 4.6 + Converse 写一个 weather agent，跟第 3 课的 ReAct 做对照
- IAM / region / model id 这些坑一次讲清